In [1]:
#importing 
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from datasets import Dataset as HFDataset
import evaluate
import pandas as pd
import numpy as np
import torch

C:\Users\Acer\OneDrive\Desktop\FRD model\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Load processed data ───────────────────────────────────────
syn_train = pd.read_csv("../data/processed/syn_train.csv")
syn_val   = pd.read_csv("../data/processed/syn_val.csv")
syn_test  = pd.read_csv("../data/processed/syn_test.csv")

In [3]:
#Model selection
MODEL_NAME = "xlm-roberta-base"
# MODEL_NAME = "bert-base-multilingual-cased"


In [4]:
# ── Detect GPU & set memory-efficient dtype ────────────────────
device    = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16  = torch.cuda.is_available()   # auto-enable on GPU
print(f"Using device : {device}")
print(f"FP16 enabled : {use_fp16}")
print(f"Model        : {MODEL_NAME}\n")

Using device : cuda
FP16 enabled : True
Model        : xlm-roberta-base



In [5]:
# ── Tokenizer ─────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        padding="max_length",
        max_length=128          # 128 saves memory vs 512; increase if needed
    )

In [6]:
# ── Dataset helper ─────────────────────────────────────────────
def to_hf_dataset(df):
    return HFDataset.from_dict({
        "review_text": df["review_text"].tolist(),
        "label":       df["label"].tolist(),
        "word_count":  df["word_count"].tolist(),
        "rating":      df["rating"].tolist(),
        "slang_count": df["slang_count"].tolist(),
    })

train_hf = to_hf_dataset(syn_train).map(tokenize, batched=True)
val_hf   = to_hf_dataset(syn_val).map(tokenize,   batched=True)
test_hf  = to_hf_dataset(syn_test).map(tokenize,  batched=True)

Map: 100%|██████████████████████████████████████████████████████████████████| 448/448 [00:00<00:00, 8848.03 examples/s]


In [7]:
# ── Model ──────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# ── Metrics ────────────────────────────────────────────────────
f1_metric  = evaluate.load("f1")
acc_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds          = np.argmax(logits, axis=1)
    return {
        "f1":       f1_metric.compute(
                        predictions=preds, references=labels,
                        average="weighted")["f1"],
        "accuracy": acc_metric.compute(
                        predictions=preds, references=labels)["accuracy"],
    }

Loading weights: 100%|█████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 4225.97it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downst

In [8]:
# ── Early stopping callback ────────────────────────────────────
# Stops training if F1 doesn't improve for 3 consecutive eval epochs
early_stop = EarlyStoppingCallback(
    early_stopping_patience  = 3,
    early_stopping_threshold = 0.001   # minimum delta to count as improvement
)

In [9]:
# ── PRETRAIN on synthetic data ─────────────────────────────────
pretrain_args = TrainingArguments(
    output_dir              = "./pretrain_checkpoints",

    # ── Epochs & batch size ───────────────────────────────────
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,   # reduce to 8 if OOM
    per_device_eval_batch_size  = 32,

    # ── Optimizer & scheduler ─────────────────────────────────
    learning_rate    = 2e-5,
    weight_decay     = 0.01,
    warmup_ratio     = 0.1,             # 10% of steps for warmup
    lr_scheduler_type= "cosine",        # cosine decay → better F1 than linear

    # ── Memory optimizations ──────────────────────────────────
    fp16                        = use_fp16,         # half-precision on GPU
    gradient_accumulation_steps = 2,                # effective batch = 16×2=32
    gradient_checkpointing      = True,             # trades compute for memory
    dataloader_pin_memory       = True,             # faster GPU data transfer
    dataloader_num_workers      = 4,                # parallel data loading

    # ── Evaluation & checkpointing ────────────────────────────
    eval_strategy     = "epoch",
    save_strategy           = "epoch",
    save_total_limit        = 3,        # keep only 3 best checkpoints on disk
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1",
    greater_is_better       = True,

    # ── Logging ───────────────────────────────────────────────
    logging_dir             = "./logs",
    logging_steps           = 50,
    report_to               = "none",   # change to "wandb" if you use W&B
)

pretrain_trainer = Trainer(
    model           = model,
    args            = pretrain_args,
    train_dataset   = train_hf,
    eval_dataset    = val_hf,
    compute_metrics = compute_metrics,
    callbacks       = [early_stop],     # ← early stopping wired in
)

print("Starting pretraining on synthetic data...")
pretrain_trainer.train()
pretrain_trainer.save_model("../models/transformers/pretrained_model")
tokenizer.save_pretrained("../models/transformers/pretrained_model")  # save tokenizer alongside model
print("Pretraining complete. Model saved to ../models/transformers/pretrained_model\n")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting pretraining on synthetic data...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.208035,0.267804,0.921710,0.921700
2,0.247305,0.066998,0.986579,0.986577
3,0.146516,0.042717,0.993289,0.993289
4,0.099996,0.023238,0.995526,0.995526
5,0.067352,0.028415,0.995526,0.995526
6,0.047837,0.027803,0.995526,0.995526
7,0.040412,0.024073,0.995526,0.995526


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.99s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.a

Pretraining complete. Model saved to ../models/transformers/pretrained_model

